# Detection and Tracking: Every Clip to an Annotated MP4

Runs `human_vehicle.tracking.track_video` over every clip in `Videos/`, renumbers the track ids
with `human_vehicle.labels.relabel_tracks`, and writes an annotated copy of each through
`human_vehicle.overlay.render_tracked_video`.

The output is an mp4 per clip at the original resolution and frame rate, with each person and
vehicle marked at the corners of its box and labelled with its category letter and its number —
`P29` is person 29 and `V4` is vehicle 4. Those are the only two classes: a car, a motorcycle, a
bus and a truck are all `V`. The numbering is 1-based and independent per class, so `P5` and `V5`
have nothing to do with each other.

It is meant to be read by a vision-language model: the scene and the tracking are visible at the
same time, so the model can be asked about either. The labels are small because every pixel the
overlay covers is a pixel the model cannot see.

The detector, the tracker and the ReID model are each a plain string, and all three are recorded in
the output filename, so runs under different configurations sit beside each other rather than
overwriting one another.

The work is all in `src/human_vehicle/`. This notebook is just the loop over the clips.

## 1. Setup

The two model choices:

- **`WEIGHTS`** — any Ultralytics detector. `yolo26n.pt` through `yolo26x.pt`, smallest and
  fastest to largest and most accurate.
- **`REID`** — a model such as `"yolo26x-reid.onnx"`, which runs as a separate encoder over every
  detection crop; `"auto"` to re-identify objects from the detector's own backbone features
  instead, with no second network to run; or `"none"`.

The tracker is left at `track_video`'s default, TrackTrack, which has the ReID stage the choice
above needs. It is still recorded in every output filename, so a run under a different one would
sit beside these rather than over them.

ReID is what stops a track id being retired and re-issued every time something is briefly occluded,
which is the failure that makes id counts overstate how many objects a clip really contains.

One tracker setting is moved off its default, and it is about holding an id through an
occlusion rather than about what gets detected:

- **`BUFFER_SECONDS`** — how long a lost track stays re-findable. Ultralytics counts this in
  frames and never scales it, so its default of 30 is one second on these 30 fps clips and five
  on the 6 fps ones; `track_video` takes the duration instead and derives the frames per clip.

`SHOW_CONFIDENCE` is a debugging view rather than a tracker setting: it writes each box's
detection confidence beside its id, as `P7 0.531`.

A render with this on is written under a `__debug` name, so it lands beside the clean clip for
the same tracking run rather than on top of it. The suffix is added where the output path is
built rather than inside `config_slug`, because the slug names the tracking configuration and
this is a property of the rendering.

**Leave it `False` for any clip being handed to a vision-language model.** The labels are four
times as wide, which is spent straight out of the pixel budget the overlay exists to respect.

The README describes what each setting does.

In [ ]:
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from human_vehicle.device import select_device
from human_vehicle.labels import relabel_tracks
from human_vehicle.overlay import render_tracked_video
from human_vehicle.tracking import Category, config_slug, track_video


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root (no pyproject.toml above the cwd).")


REPO_ROOT = find_repo_root()
VIDEOS_DIR = REPO_ROOT / "Videos"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "tracked"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = "yolo26x.pt"  # largest variant; downloads on first use
REID = "yolo26x-reid.onnx"  # a separate encoder, matched to the detector; downloads on first use
CRF = 15  # visually near-lossless for x264
SHOW_CONFIDENCE = False  # draw each box's confidence next to its id; see the note above

BUFFER_SECONDS = 3.0  # 20 frames on the 6 fps clips, 90 on the 30 fps ones

print(f"repo root: {REPO_ROOT}")
print(f"clips in:  {VIDEOS_DIR}")
print(f"mp4s out:  {OUTPUT_DIR}")
print(f"config:    {WEIGHTS} + reid={REID} on {select_device()}")
print(f"tracking:  buffer={BUFFER_SECONDS}s")
print(f"overlay:   crf={CRF} confidence={SHOW_CONFIDENCE}")

## 2. The clips

`Videos/` is git-ignored local data, so this is whatever you have put there.

The probe is here rather than after the fact because the detector's input size has to be chosen per
clip, and these range from 352x288 to 4K. Ultralytics scales a frame's longest side to `imgsz`.

640 px everywhere except 4K footage, which gets 1280. Running the sub-HD clips at their native
scale was tried and measured — on `iMGR_0AG3a8_2_3` it found about 10% more detections at higher
mean confidence — but the extra objects were not worth two to four times the runtime here, so the
cheaper setting stands.

In [ ]:
def probe(path: Path) -> dict[str, int]:
    """Frame size and count, straight from the decoder."""
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise RuntimeError(f"could not open {path}")
    info = {
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frames": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    capture.release()
    return info


def imgsz_for(height: int) -> int:
    """Raise the detector's input size on tall frames, where 640 px loses distant people."""
    return 1280 if height >= 1080 else 640


CLIPS = {path.stem: {"path": path, **probe(path)} for path in sorted(VIDEOS_DIR.glob("*.mp4"))}
if not CLIPS:
    raise RuntimeError(f"no mp4 files in {VIDEOS_DIR}")

for clip_id, info in CLIPS.items():
    size = f"{info['width']}x{info['height']}"
    print(f"{clip_id:<24} {size:>11} {info['frames']:>4} frames  imgsz={imgsz_for(info['height'])}")

## 3. Track, renumber, then render

Tracking and rendering are separate passes on purpose: tracking produces a small record of boxes
and ids that stays in memory, and rendering reads the clip a second time to draw it. Keeping them
apart means the annotated video can be re-rendered — different colors, a different CRF — without
paying for detection again.

Renumbering sits between them. The tracker issues ids across all classes at once, so its people
come out as `P3`, `P17`, `P42`; `relabel_tracks` makes each category count from 1 without gaps,
which is what the downstream task reads. It also hands back the translation from the original
`(category, track_id)` identities, kept here per clip and shown in section 5.

**This takes a while.** YOLO26x is the largest variant, the 4K clip runs at `imgsz=1280`, and the
ReID encoder is a second network run over every detection crop. A long-running cell here is
normal, not a hang.

Each output is named `{clip_id}__{config}.mp4`, where the configuration comes from the record the
run produced rather than from the constants above — so the filename cannot claim a configuration
that is not the one that made it. A render with `SHOW_CONFIDENCE` on gets a further `__debug`,
which keeps it clear of the clean clip for the same tracking run.

In [ ]:
RESULTS: dict[str, dict[str, object]] = {}
TRANSLATIONS: dict[str, dict[tuple[Category, int], str]] = {}

for clip_id, info in CLIPS.items():
    print(f"{clip_id} ... ", end="", flush=True)

    started = time.perf_counter()
    tracks = track_video(
        info["path"],
        weights=WEIGHTS,
        reid=REID,
        imgsz=imgsz_for(info["height"]),
        buffer_seconds=BUFFER_SECONDS,
    )
    tracked_at = time.perf_counter()
    relabelled, translation = relabel_tracks(tracks)
    # `config_slug` names the tracking configuration and knows nothing about rendering, so
    # the overlay's own setting is marked here rather than folded into it.
    debug = "__debug" if SHOW_CONFIDENCE else ""
    output = OUTPUT_DIR / f"{clip_id}__{config_slug(relabelled)}{debug}.mp4"
    render_tracked_video(relabelled, output, crf=CRF, show_confidence=SHOW_CONFIDENCE)
    finished = time.perf_counter()

    # One entry per identity the tracker found, so this counts objects rather than detections.
    TRANSLATIONS[clip_id] = translation
    people = sum(1 for category, _ in translation if category is Category.PERSON)
    RESULTS[clip_id] = {
        "output": output,
        "width": relabelled.width,
        "height": relabelled.height,
        "fps": relabelled.fps,
        "frames": len(relabelled.frames),
        "people": people,
        "vehicles": len(translation) - people,
        "track_s": tracked_at - started,
        "render_s": finished - tracked_at,
        "mb": output.stat().st_size / 1e6,
    }
    row = RESULTS[clip_id]
    print(f"people={row['people']} vehicles={row['vehicles']} in {finished - started:.1f}s")

print(f"\nwrote {len(RESULTS)} mp4 files to {OUTPUT_DIR}")

## 4. What came out

`frames` and `fps` are the source's, carried through exactly — the annotated clip is frame-for-frame
the original, so a timestamp in the output means the same thing as a timestamp in the input.

`people` and `vehicles` count distinct identities, not detections, so they are also the highest `P`
and `V` numbers drawn on each clip. They are an upper bound on the number of real objects: an id is
lost and re-issued whenever something leaves the frame or is occluded for longer than the tracker's
buffer, so a busy clip inflates them. ReID pulls these numbers back towards the truth, so they
should be lower here than with `REID = "none"`.

Watch for the opposite failure too: a count that drops because two different objects were merged
into one identity looks exactly like a count that drops because a fragmented track was correctly
rejoined. Only the video tells them apart.

In [ ]:
header = (
    f"{'clip':<24} {'size':>11} {'fps':>10} {'frames':>7} {'people':>7} "
    f"{'vehicles':>9} {'track':>7} {'render':>7} {'out':>8}"
)
print(header)
print("-" * len(header))
for clip_id, row in RESULTS.items():
    print(
        f"{clip_id:<24} {f'{row["width"]}x{row["height"]}':>11} {row['fps']:>10} {row['frames']:>7} "
        f"{row['people']:>7} {row['vehicles']:>9} {row['track_s']:>6.1f}s {row['render_s']:>6.1f}s {row['mb']:>6.1f}MB"
    )

## 5. Original ids to labels

The translation for one clip: what the tracker called each object on the left, what the video calls
it on the right. This is the only place the original ids survive, and nothing downstream needs them
— it is here so a label in the video can be traced back to the tracker's own output when something
looks wrong.

Note how sparse and interleaved the left-hand column is. That is what the renumbering fixes.

`TRANSLATIONS` holds one of these per clip, as a plain dict keyed by `(category, track_id)`.
Saving a run's mappings is a `json.dumps` over string keys — `f"{category.value}{track_id}"` — if
a debugging session is worth keeping.

In [ ]:
CLIP = next(iter(TRANSLATIONS))
translation = TRANSLATIONS[CLIP]

print(f"{CLIP}: {len(translation)} identities\n")
for (category, track_id), label in sorted(translation.items(), key=lambda item: (item[0][0], int(item[1][1:]))):
    print(f"  {category.value:<7} {track_id:>4}  ->  {label}")

## 6. Spot check

A frame from the middle of each annotated clip, downscaled for display only. This is the check that
matters before handing anything to a model: the labels have to be legible, and it has to be obvious
which box each one belongs to.

Person boxes are azure and vehicle boxes orange, so the two read apart at a glance without reading
the `P` or the `V`. A box is marked at its corners rather than outlined, which fixes its extent
without covering the object. Labels are placed off each other, each staying attached to one of those
corners; they overlap only where a frame is crowded enough that a label has nowhere free to go,
which is worth knowing if a model later misreads a dense scene.

In [ ]:
def middle_frame(path: Path, *, display_width: int = 640):
    """The frame halfway through a clip, as RGB, scaled down to display width."""
    capture = cv2.VideoCapture(str(path))
    capture.set(cv2.CAP_PROP_POS_FRAMES, capture.get(cv2.CAP_PROP_FRAME_COUNT) // 2)
    decoded, frame = capture.read()
    capture.release()
    if not decoded:
        raise RuntimeError(f"could not read a frame from {path}")
    height = round(frame.shape[0] * display_width / frame.shape[1])
    return cv2.cvtColor(cv2.resize(frame, (display_width, height)), cv2.COLOR_BGR2RGB)


columns = 2
rows = -(-len(RESULTS) // columns)
figure, axes = plt.subplots(rows, columns, figsize=(14, 4.2 * rows))
for axis, (clip_id, row) in zip(axes.ravel(), RESULTS.items(), strict=False):
    axis.imshow(middle_frame(row["output"]))
    axis.set_title(f"{clip_id}  ({row['people']} people, {row['vehicles']} vehicles)", fontsize=9)
for axis in axes.ravel():
    axis.axis("off")
figure.tight_layout()